# Edge AI Pipeline — MiniLM-L12 × AG News

**Master's project | Data Science & AI**

| Block | Purpose |
|---|---|
| 1 | Setup, Dataset Loading & Preprocessing |
| 2 | Optimizer Benchmark Training Loop (AdamW · Adafactor · Lion · LAMB · SGD) |
| 3 | Post-Training Compression — Dynamic INT8 Quantization |
| 4 | Edge AI Evaluation & Latency Benchmarking |
| 5 | Visualization — Futuristic Dark Theme |

---


## 🔧 Installation
Run the cell below once (Colab / local venv).


In [ ]:
# ── Step 1: Install PyTorch WITH CUDA support ──────────────────────────
# IMPORTANT: the default "pip install torch" gives the CPU-only build.
# Run the line matching your CUDA version (driver reports CUDA 13.1 here).
#
# CUDA 12.6 (recommended for driver >= 525, CUDA <= 13.x)
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
#
# CUDA 12.4 (alternative)
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
#
# CPU-only fallback (slow, not recommended for training)
# !pip install torch torchvision torchaudio

# ── Step 2: Other dependencies ──────────────────────────────────────────
# !pip install -q transformers datasets evaluate scikit-learn
# !pip install -q psutil accelerate
# !pip install -q lion-pytorch      # Lion optimizer
# !pip install -q torch-optimizer   # LAMB optimizer
# !pip install -q matplotlib


---
## Block 1 — Setup, Dataset Loading & Preprocessing

- Loads `ag_news` (4-class topic classification) from Hugging Face Hub  
- Tokenizes with `microsoft/MiniLM-L12-H384-uncased`  
- Returns `DatasetDict` of `torch` tensors ready for the Trainer  


In [1]:
import os
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer

# ── Global configuration ─────────────────────────────────────────────────
MODEL_NAME    = "microsoft/MiniLM-L12-H384-uncased"
DATASET_NAME  = "ag_news"
MAX_LENGTH    = 128
SEED          = 42
TRAIN_SAMPLES = 10_000   # set None for full ~120 k
EVAL_SAMPLES  =  2_000   # set None for full   ~7.6 k
OUTPUT_DIR           = "./results"
BEST_MODEL_DIR       = "./best_model"
COMPRESSED_MODEL_DIR = "./compressed_model"
LABEL_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
NUM_LABELS  = 4

for _d in (OUTPUT_DIR, BEST_MODEL_DIR, COMPRESSED_MODEL_DIR):
    os.makedirs(_d, exist_ok=True)


# ── Device diagnostic ────────────────────────────────────────────────────
print("=" * 55)
print(f"  PyTorch version : {torch.__version__}")
print(f"  CUDA available  : {torch.cuda.is_available()}")
print(f"  CUDA built-in   : {torch.version.cuda}")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"  GPU device      : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM total      : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"  VRAM free       : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3:.1f} GB")
else:
    DEVICE = torch.device("cpu")
    print()
    print("  [!] CPU-only PyTorch detected.")
    print("  [!] To enable your GPU, reinstall PyTorch with CUDA support:")
    print("  [!]   pip uninstall torch torchvision torchaudio -y")
    print("  [!]   pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126")
    print("  [!] Then restart the kernel and re-run this cell.")

print(f"  Active device   : {DEVICE}")
print("=" * 55)


c:\Users\HP ZBOOK\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


  PyTorch version : 2.12.1+cu126
  CUDA available  : True
  CUDA built-in   : 12.6
  GPU device      : Quadro T1000
  VRAM total      : 4.0 GB
  VRAM free       : 4.0 GB
  Active device   : cuda


In [2]:
def load_and_preprocess_data():
    """Load AG News, subsample, tokenize, return DatasetDict + tokenizer."""
    print('[1/3] Loading dataset...')
    dataset = load_dataset(DATASET_NAME)
    if TRAIN_SAMPLES:
        dataset['train'] = dataset['train'].shuffle(seed=SEED).select(range(TRAIN_SAMPLES))
    if EVAL_SAMPLES:
        dataset['test']  = dataset['test'].shuffle(seed=SEED).select(range(EVAL_SAMPLES))
    print(f'   Train: {len(dataset["train"]):,}  |  Test: {len(dataset["test"]):,}')

    print(f'[2/3] Loading tokenizer: {MODEL_NAME}')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize_fn(batch):
        return tokenizer(batch['text'], padding='max_length',
                         truncation=True, max_length=MAX_LENGTH)

    print('[3/3] Tokenizing...')
    tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
    tokenized = tokenized.rename_column('label', 'labels')
    tokenized.set_format('torch')
    print('[OK] Done.\n')
    return tokenized, tokenizer


tokenized_dataset, tokenizer = load_and_preprocess_data()


[1/3] Loading dataset...
   Train: 10,000  |  Test: 2,000
[2/3] Loading tokenizer: microsoft/MiniLM-L12-H384-uncased
[3/3] Tokenizing...


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

[OK] Done.



---
## Block 2 — Optimizer Benchmark Training Loop

Iterates over **5 optimizers** using the HuggingFace `Trainer` API.  
Tracks per optimizer: Accuracy · Macro-F1 · Training time · CPU/GPU peak memory · Loss stability.

| Optimizer | Notes |
|---|---|
| AdamW | Standard baseline |
| Adafactor | Self-adaptive LR, memory-efficient |
| Lion | Sign-based momentum; uses lr×0.1 |
| LAMB | Layer-wise LR adaptation (large-batch) |
| SGD | Nesterov momentum baseline |


In [3]:
import time, json, psutil
import evaluate
from transformers import (
    AutoModelForSequenceClassification, TrainingArguments, Trainer,
    TrainerCallback, get_linear_schedule_with_warmup,
)
from transformers.optimization import Adafactor, AdafactorSchedule
from torch.optim import SGD, AdamW
import torch.nn as nn

# Optional optimizers
try:
    from lion_pytorch import Lion;  HAS_LION = True
except ImportError:
    HAS_LION = False;  print('[WARN] pip install lion-pytorch')
try:
    import torch_optimizer as topt; HAS_LAMB = True
except ImportError:
    HAS_LAMB = False;  print('[WARN] pip install torch-optimizer')

# ── Metrics ──────────────────────────────────────────────────────────────
_acc = evaluate.load('accuracy')
_f1  = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': _acc.compute(predictions=preds, references=labels)['accuracy'],
        'f1_macro': _f1.compute(predictions=preds, references=labels, average='macro')['f1'],
    }

# ── Memory Callback ───────────────────────────────────────────────────────
class MemoryTrackerCallback(TrainerCallback):
    def __init__(self, every=20):
        self.every = every; self._step = 0
        self.cpu_mb = []; self.gpu_mb = []
    def on_step_end(self, args, state, control, **kw):
        self._step += 1
        if self._step % self.every: return
        self.cpu_mb.append(psutil.Process().memory_info().rss / 1024**2)
        if torch.cuda.is_available():
            self.gpu_mb.append(torch.cuda.max_memory_allocated() / 1024**2)
    @property
    def peak_cpu(self): return max(self.cpu_mb, default=0.0)
    @property
    def peak_gpu(self): return max(self.gpu_mb, default=0.0)


In [4]:
# ── Optimizer & Scheduler Factory ────────────────────────────────────────
def _param_groups(model, wd=0.01):
    no_decay = {'bias', 'LayerNorm.weight'}
    return [
        {'params': [p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': wd},
        {'params': [p for n,p in model.named_parameters() if     any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
    ]

def build_optimizer(name, model, lr, n_train, epochs, batch):
    n_steps  = (n_train // batch) * epochs
    n_warmup = max(1, n_steps // 10)
    pg = _param_groups(model)

    if name == 'AdamW':
        opt = AdamW(pg, lr=lr, eps=1e-8)
        sch = get_linear_schedule_with_warmup(opt, n_warmup, n_steps)

    elif name == 'Adafactor':
        opt = Adafactor(model.parameters(), lr=None,
                        scale_parameter=True, relative_step=True, warmup_init=True)
        sch = AdafactorSchedule(opt)

    elif name == 'Lion':
        if not HAS_LION: raise ImportError('pip install lion-pytorch')
        opt = Lion(pg, lr=lr * 0.1, weight_decay=0.01)
        sch = get_linear_schedule_with_warmup(opt, n_warmup, n_steps)

    elif name == 'LAMB':
        if not HAS_LAMB: raise ImportError('pip install torch-optimizer')
        opt = topt.Lamb(pg, lr=lr, weight_decay=0.01)
        sch = get_linear_schedule_with_warmup(opt, n_warmup, n_steps)

    elif name == 'SGD':
        flat = [p for p in model.parameters() if p.requires_grad]
        opt = SGD(flat, lr=lr, momentum=0.9, weight_decay=1e-4, nesterov=True)
        sch = get_linear_schedule_with_warmup(opt, n_warmup, n_steps)

    else:
        raise ValueError(name)
    return opt, sch


In [5]:
# ── Single-optimizer training run ────────────────────────────────────────
def train_with_optimizer(opt_name, lr=2e-5, epochs=3, batch=32):
    print(f'\n{"━"*55}\n  Optimizer: {opt_name}\n{"━"*55}')
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
    optimizer, scheduler = build_optimizer(
        opt_name, model, lr, len(tokenized_dataset['train']), epochs, batch)
    mem_cb = MemoryTrackerCallback()

    args = TrainingArguments(
        output_dir                  = f'{OUTPUT_DIR}/{opt_name}',
        num_train_epochs            = epochs,
        per_device_train_batch_size = batch,
        per_device_eval_batch_size  = 64,
        evaluation_strategy         = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'f1_macro',
        greater_is_better           = True,
        logging_steps               = 50,
        report_to                   = 'none',
        seed                        = SEED,
        fp16                        = torch.cuda.is_available(),
        dataloader_num_workers      = 0,
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=tokenized_dataset['train'],
        eval_dataset=tokenized_dataset['test'],
        compute_metrics=compute_metrics,
        optimizers=(optimizer, scheduler),
        callbacks=[mem_cb],
    )

    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    ev = trainer.evaluate()

    logs   = trainer.state.log_history
    tl     = [x['loss']          for x in logs if 'loss'          in x and 'eval_loss' not in x]
    el     = [x['eval_loss']     for x in logs if 'eval_loss'     in x]
    ea     = [x['eval_accuracy'] for x in logs if 'eval_accuracy' in x]
    ef1    = [x['eval_f1_macro'] for x in logs if 'eval_f1_macro' in x]

    result = {
        'optimizer':          opt_name,
        'final_accuracy':     ev['eval_accuracy'],
        'final_f1_macro':     ev['eval_f1_macro'],
        'training_time_s':    round(elapsed, 2),
        'peak_cpu_memory_mb': round(mem_cb.peak_cpu, 2),
        'peak_gpu_memory_mb': round(mem_cb.peak_gpu, 2),
        'train_losses':       tl,
        'eval_losses':        el,
        'eval_accuracies':    ea,
        'eval_f1s':           ef1,
        'loss_std': float(np.std(tl[len(tl)//2:])) if tl else 0.0,
    }
    print(f'  Accuracy : {result["final_accuracy"]:.4f}')
    print(f'  F1 macro : {result["final_f1_macro"]:.4f}')
    print(f'  Time     : {elapsed:.1f} s')
    print(f'  CPU RAM  : {mem_cb.peak_cpu:.0f} MB')
    return result, trainer


In [ ]:
# ── Master benchmark loop ────────────────────────────────────────────────
OPTIMIZERS = ['AdamW', 'Adafactor', 'Lion', 'LAMB', 'SGD']
LR, EPOCHS, BATCH = 2e-5, 3, 32

all_results, best_result, best_trainer = [], None, None
best_f1 = -1.0

for opt in OPTIMIZERS:
    try:
        res, tr = train_with_optimizer(opt, lr=LR, epochs=EPOCHS, batch=BATCH)
        all_results.append(res)
        if res['final_f1_macro'] > best_f1:
            best_f1, best_result, best_trainer = res['final_f1_macro'], res, tr
    except ImportError as e:
        print(f'[SKIP] {opt}: {e}')
    except Exception as e:
        print(f'[ERROR] {opt}: {e}')

# Save best checkpoint & results JSON
if best_trainer:
    best_trainer.save_model(BEST_MODEL_DIR)
    print(f'Best model: {best_result["optimizer"]} | F1={best_f1:.4f} → {BEST_MODEL_DIR}')

with open(f'{OUTPUT_DIR}/benchmark_results.json', 'w') as fh:
    json.dump(all_results, fh, indent=2)

# Summary table
print(f'\n{"Optimizer":<12} {"Accuracy":>9} {"F1":>9} {"Time(s)":>9} {"CPU MB":>9}')
print('-'*50)
for r in sorted(all_results, key=lambda x: x['final_f1_macro'], reverse=True):
    mark = ' ★' if r['optimizer'] == best_result['optimizer'] else ''
    print(f"{r['optimizer']:<12} {r['final_accuracy']:>9.4f} {r['final_f1_macro']:>9.4f}"
          f" {r['training_time_s']:>9.1f} {r['peak_cpu_memory_mb']:>9.0f}{mark}")



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Optimizer: AdamW
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/MiniLM-L12-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\HP ZBOOK\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


  0%|          | 0/939 [00:00<?, ?it/s]

{'loss': 1.3857, 'grad_norm': 0.4127894341945648, 'learning_rate': 1.0752688172043012e-05, 'epoch': 0.16}
{'loss': 1.122, 'grad_norm': 4.565655708312988, 'learning_rate': 1.9833926453143536e-05, 'epoch': 0.32}
{'loss': 0.6665, 'grad_norm': 3.344130516052246, 'learning_rate': 1.8647686832740214e-05, 'epoch': 0.48}
{'loss': 0.5279, 'grad_norm': 10.765714645385742, 'learning_rate': 1.7461447212336893e-05, 'epoch': 0.64}
{'loss': 0.4612, 'grad_norm': 4.949830055236816, 'learning_rate': 1.627520759193357e-05, 'epoch': 0.8}
{'loss': 0.4104, 'grad_norm': 2.669376850128174, 'learning_rate': 1.508896797153025e-05, 'epoch': 0.96}


  0%|          | 0/32 [00:00<?, ?it/s]

{'eval_loss': 0.39490270614624023, 'eval_accuracy': 0.891, 'eval_f1_macro': 0.8919426435157144, 'eval_runtime': 42.8395, 'eval_samples_per_second': 46.686, 'eval_steps_per_second': 0.747, 'epoch': 1.0}
{'loss': 0.3701, 'grad_norm': 2.07646107673645, 'learning_rate': 1.3902728351126929e-05, 'epoch': 1.12}
{'loss': 0.3585, 'grad_norm': 4.073887348175049, 'learning_rate': 1.2716488730723606e-05, 'epoch': 1.28}
{'loss': 0.3296, 'grad_norm': 2.5687830448150635, 'learning_rate': 1.1530249110320286e-05, 'epoch': 1.44}
{'loss': 0.3425, 'grad_norm': 6.543051242828369, 'learning_rate': 1.0344009489916964e-05, 'epoch': 1.6}
{'loss': 0.3337, 'grad_norm': 5.341185092926025, 'learning_rate': 9.157769869513643e-06, 'epoch': 1.76}
{'loss': 0.3131, 'grad_norm': 5.340475082397461, 'learning_rate': 7.971530249110322e-06, 'epoch': 1.92}


  0%|          | 0/32 [00:00<?, ?it/s]

{'eval_loss': 0.3352373540401459, 'eval_accuracy': 0.8975, 'eval_f1_macro': 0.8984176786549638, 'eval_runtime': 42.3989, 'eval_samples_per_second': 47.171, 'eval_steps_per_second': 0.755, 'epoch': 2.0}


---
## Block 3 — Post-Training Compression: Dynamic INT8 Quantization

Applies `torch.quantization.quantize_dynamic` to all `nn.Linear` layers.

- **No calibration data** required (dynamic = activations quantized at runtime)  
- Weights stored as `qint8` → ~2–4× smaller model, ~1.5–2× faster CPU inference  
- Typical accuracy drop < 0.5% on classification tasks  


In [ ]:
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig

def get_model_size_mb(model):
    """Sum parameter + buffer bytes (works for FP32 and INT8 models)."""
    total = sum(t.nelement() * t.element_size()
                for t in (*model.parameters(), *model.buffers()))
    return total / 1024**2

def get_disk_size_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / 1024**2

def apply_dynamic_quantization(model_dir=BEST_MODEL_DIR, output_dir=COMPRESSED_MODEL_DIR):
    """Quantize all Linear layers to INT8 and persist the result."""
    print('[1/4] Loading FP32 baseline...')
    baseline = AutoModelForSequenceClassification.from_pretrained(
        model_dir, num_labels=NUM_LABELS).eval()

    print('[2/4] Applying quantize_dynamic (INT8, Linear layers)...')
    quantized = torch.quantization.quantize_dynamic(
        baseline, {nn.Linear}, dtype=torch.qint8).eval()

    b_mb = get_model_size_mb(baseline)
    q_mb = get_model_size_mb(quantized)
    print(f'  FP32 size : {b_mb:.2f} MB')
    print(f'  INT8 size : {q_mb:.2f} MB  ({b_mb/q_mb:.2f}× compression)')

    print(f'[3/4] Saving to {output_dir}...')
    os.makedirs(output_dir, exist_ok=True)
    torch.save(quantized.state_dict(), f'{output_dir}/quantized_model.pt')
    baseline.config.save_pretrained(output_dir)
    AutoTokenizer.from_pretrained(MODEL_NAME).save_pretrained(output_dir)

    bd = get_disk_size_mb(model_dir)
    qd = get_disk_size_mb(output_dir)
    print(f'[4/4] Disk — Baseline: {bd:.1f} MB  |  Quantized: {qd:.1f} MB')
    print('[OK] Quantization complete.')
    return quantized, baseline

def load_quantized_model(model_dir=COMPRESSED_MODEL_DIR):
    cfg   = AutoConfig.from_pretrained(model_dir)
    shell = AutoModelForSequenceClassification.from_config(cfg)
    shell = torch.quantization.quantize_dynamic(shell, {nn.Linear}, dtype=torch.qint8)
    shell.load_state_dict(torch.load(f'{model_dir}/quantized_model.pt', map_location='cpu'))
    return shell.eval()


quantized_model, baseline_model = apply_dynamic_quantization()


---
## Block 4 — Edge AI Evaluation & Latency Benchmarking

Compares **FP32 baseline** vs **INT8 quantized** model on:

| Metric | What it reveals |
|---|---|
| Disk size (MB) | Storage budget on device |
| In-memory size (MB) | RAM footprint |
| Compression ratio | How much smaller |
| Latency mean / std / P95 (ms) | Responsiveness & jitter |
| Speedup | Inference throughput gain |
| Accuracy / F1 drop | Quality cost of compression |


In [ ]:
def measure_latency(model, tokenizer, text, device, n_warmup=20, n_runs=200):
    """Single-example latency (batch=1) — returns mean/std/P95 in ms."""
    model.eval().to(device)
    enc = tokenizer(text, return_tensors='pt', padding='max_length',
                    truncation=True, max_length=MAX_LENGTH)
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        for _ in range(n_warmup): model(**enc)
    lats = []
    with torch.no_grad():
        for _ in range(n_runs):
            if device.type == 'cuda': torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(**enc)
            if device.type == 'cuda': torch.cuda.synchronize()
            lats.append((time.perf_counter() - t0) * 1000)
    a = np.array(lats)
    return {'mean_ms': float(a.mean()), 'std_ms': float(a.std()),
            'p95_ms':  float(np.percentile(a, 95))}


def eval_accuracy(model, tokenizer, dataset, device, batch=64):
    model.eval().to(device)
    lbl_col = 'label' if 'label' in dataset.column_names else 'labels'
    preds, labels = [], []
    for i in range(0, len(dataset), batch):
        enc = tokenizer(dataset['text'][i:i+batch], return_tensors='pt',
                        padding=True, truncation=True, max_length=MAX_LENGTH)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
        preds  += torch.argmax(logits, -1).cpu().tolist()
        batch_labels = dataset[lbl_col][i:i+batch]
        labels += (batch_labels.tolist() if hasattr(batch_labels, 'tolist')
                   else list(batch_labels))
    acc = _acc.compute(predictions=preds, references=labels)['accuracy']
    f1  = _f1.compute( predictions=preds, references=labels, average='macro')['f1']
    return {'accuracy': acc, 'f1_macro': f1}


In [ ]:
def full_edge_evaluation(eval_samples=500, n_runs=200):
    cpu = torch.device('cpu')   # edge = CPU inference
    print('\n' + '='*65)
    print('  BLOCK 4 — Edge AI Evaluation Report')
    print('='*65)

    tok   = AutoTokenizer.from_pretrained(MODEL_NAME)
    raw   = load_dataset('ag_news', split='test').shuffle(seed=42).select(range(eval_samples))
    b_mdl = AutoModelForSequenceClassification.from_pretrained(
                BEST_MODEL_DIR, num_labels=NUM_LABELS).eval()
    q_mdl = load_quantized_model()

    # Size
    b_disk = get_disk_size_mb(BEST_MODEL_DIR)
    q_disk = get_disk_size_mb(COMPRESSED_MODEL_DIR)
    b_mem  = get_model_size_mb(b_mdl)
    q_mem  = get_model_size_mb(q_mdl)

    # Latency (CPU)
    print('Measuring latency...')
    b_lat = measure_latency(b_mdl, tok, raw['text'][0], cpu, n_runs=n_runs)
    q_lat = measure_latency(q_mdl, tok, raw['text'][0], cpu, n_runs=n_runs)
    spdup = b_lat['mean_ms'] / q_lat['mean_ms']

    # Accuracy
    print('Computing accuracy...')
    b_met = eval_accuracy(b_mdl, tok, raw, cpu)
    q_met = eval_accuracy(q_mdl, tok, raw, cpu)

    W = 40
    def row(lbl, bv, qv, fmt='.2f'):
        print(f'  {lbl:<{W}} {bv:>10{fmt}} {qv:>12{fmt}}')

    print(f'\n  {"Metric":<{W}} {"Baseline":>10} {"INT8 Quant":>12}')
    print('  ' + '-'*(W+24))
    row('Disk size (MB)',              b_disk, q_disk)
    row('In-memory size (MB)',         b_mem,  q_mem)
    print(f'  {"Compression (disk)":<{W}} {"—":>10} {b_disk/q_disk:>11.2f}x')
    print(f'  {"Compression (memory)":<{W}} {"—":>10} {b_mem/q_mem:>11.2f}x')
    print('  ' + '-'*(W+24))
    row('Latency mean (ms)',           b_lat['mean_ms'], q_lat['mean_ms'])
    row('Latency std  (ms)',           b_lat['std_ms'],  q_lat['std_ms'])
    row('Latency P95  (ms)',           b_lat['p95_ms'],  q_lat['p95_ms'])
    print(f'  {"Speedup":<{W}} {"—":>10} {spdup:>11.2f}x')
    print('  ' + '-'*(W+24))
    row('Accuracy',                   b_met['accuracy'], q_met['accuracy'], fmt='.4f')
    print(f'  {"Accuracy drop":<{W}} {"—":>10} {b_met["accuracy"]-q_met["accuracy"]:>+11.4f}')
    row('F1 Macro',                   b_met['f1_macro'], q_met['f1_macro'], fmt='.4f')
    print(f'  {"F1 drop":<{W}} {"—":>10} {b_met["f1_macro"]-q_met["f1_macro"]:>+11.4f}')
    print('  ' + '='*(W+24))

    report = {
        'baseline_disk_mb': b_disk, 'quantized_disk_mb': q_disk,
        'baseline_mem_mb':  b_mem,  'quantized_mem_mb':  q_mem,
        'compression_ratio_disk': b_disk/q_disk,
        'compression_ratio_mem':  b_mem/q_mem,
        'baseline_latency': b_lat,  'quantized_latency': q_lat,
        'speedup': spdup,
        'baseline_accuracy': b_met['accuracy'], 'quantized_accuracy': q_met['accuracy'],
        'accuracy_drop': b_met['accuracy'] - q_met['accuracy'],
        'baseline_f1':   b_met['f1_macro'], 'quantized_f1':   q_met['f1_macro'],
        'f1_drop':       b_met['f1_macro'] - q_met['f1_macro'],
    }
    with open(f'{OUTPUT_DIR}/edge_evaluation_report.json', 'w') as fh:
        json.dump(report, fh, indent=2)
    return report


eval_report = full_edge_evaluation(eval_samples=500, n_runs=200)


---
## Block 5 — Visualization

Futuristic dark theme — deep-blue backgrounds × electric-cyan palette.

| Plot | Description |
|---|---|
| 1 | Smoothed training loss — all optimizers |
| 2 | Validation loss per epoch |
| 3 | Accuracy & Macro-F1 per epoch (side by side) |
| 4 | Radar chart — 5-dim optimizer comparison |
| 5 | Memory & timing horizontal bars |
| 6 | Edge AI compression dashboard |


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
from pathlib import Path

# ── Palette ───────────────────────────────────────────────────────────────
PALETTE = {
    'AdamW':     '#00FFFF',   # electric cyan
    'Adafactor': '#1E90FF',   # dodger blue
    'Lion':      '#BF5FFF',   # electric violet
    'LAMB':      '#FF6B35',   # neon orange
    'SGD':       '#39FF14',   # neon green
}
BG       = '#050A1A'
BG_PANEL = '#080E20'
GRID_CLR = '#0D2137'
TEXT_CLR = '#D0EEF8'
ACCENT   = '#00FFFF'
BASE_CLR = '#1E90FF'
QUANT_CLR= '#00FFFF'

def apply_theme():
    matplotlib.rcParams.update({
        'figure.facecolor': BG,       'figure.dpi': 110,
        'axes.facecolor':   BG_PANEL,  'axes.edgecolor':  ACCENT,
        'axes.labelcolor':  TEXT_CLR,  'axes.titlecolor': TEXT_CLR,
        'axes.titlepad':    14,        'axes.grid':       True,
        'axes.axisbelow':   True,      'grid.color':      GRID_CLR,
        'grid.linewidth':   0.75,      'grid.linestyle':  '--',
        'xtick.color':      TEXT_CLR,  'ytick.color':     TEXT_CLR,
        'legend.facecolor': '#0A1628', 'legend.edgecolor':ACCENT,
        'legend.labelcolor':TEXT_CLR,  'legend.fontsize': 9,
        'lines.linewidth':  2.2,       'font.family':     'monospace',
        'font.size':        10,        'savefig.facecolor':BG,
        'savefig.dpi':      180,       'savefig.bbox':    'tight',
    })

def _spine(ax, c=ACCENT, lw=1.5):
    for s in ax.spines.values(): s.set_edgecolor(c); s.set_linewidth(lw)

def _smooth(v, w=7):
    a = np.array(v, float)
    if len(a) < w: return a
    return np.convolve(np.pad(a,(w//2,w//2),mode='edge'), np.ones(w)/w, 'valid')


In [ ]:
apply_theme()

# ── Plot 1: Training loss ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))
for r in all_results:
    n, tl = r['optimizer'], r.get('train_losses', [])
    if not tl: continue
    c = PALETTE.get(n, '#FFF')
    x = np.linspace(0, 1, len(tl))
    ax.plot(x, tl,          color=c, alpha=0.12, lw=1)
    ax.plot(x, _smooth(tl), color=c, label=n, lw=2.4)
ax.set_title('Training Loss — Optimizer Comparison', fontsize=14, fontweight='bold')
ax.set_xlabel('Training Progress (normalized steps)')
ax.set_ylabel('Cross-Entropy Loss'); ax.legend(); _spine(ax)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_train_loss.png'); plt.show()
print('[Plot 1 saved]')


In [ ]:
apply_theme()

# ── Plot 2: Validation loss per epoch ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
for r in all_results:
    n, el = r['optimizer'], r.get('eval_losses', [])
    if not el: continue
    c = PALETTE.get(n, '#FFF')
    ax.plot(np.arange(1, len(el)+1), el, color=c, label=n,
            marker='o', markersize=7, markerfacecolor=BG, markeredgewidth=2)
ax.set_title('Validation Loss per Epoch', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation Loss')
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(); _spine(ax)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_eval_loss.png'); plt.show()
print('[Plot 2 saved]')


In [ ]:
apply_theme()

# ── Plot 3: Accuracy & F1 side by side ───────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for r in all_results:
    n = r['optimizer']; c = PALETTE.get(n, '#FFF')
    mk = dict(markersize=8, markerfacecolor=BG, markeredgewidth=2)
    if r.get('eval_accuracies'):
        e = np.arange(1, len(r['eval_accuracies'])+1)
        ax1.plot(e, r['eval_accuracies'], color=c, label=n, marker='^', **mk)
    if r.get('eval_f1s'):
        e = np.arange(1, len(r['eval_f1s'])+1)
        ax2.plot(e, r['eval_f1s'],        color=c, label=n, marker='s', **mk)
for ax, ttl, yl in [(ax1,'Validation Accuracy','Accuracy'),(ax2,'Validation Macro-F1','F1')]:
    ax.set_title(ttl, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(yl)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend(loc='lower right'); _spine(ax)
fig.suptitle('MiniLM-L12 · AG News — Optimizer Benchmark',
             fontsize=15, fontweight='bold', color=ACCENT, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_accuracy_f1.png'); plt.show()
print('[Plot 3 saved]')


In [ ]:
apply_theme()

# ── Plot 4: Radar chart ───────────────────────────────────────────────────
dims   = ['Accuracy', 'F1 Macro', 'Speed\n(1/time)', 'CPU\nEfficiency', 'Loss\nStability']
N      = len(dims)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]

fig, ax = plt.subplots(figsize=(9,9), subplot_kw=dict(polar=True))
ax.set_facecolor(BG_PANEL); fig.patch.set_facecolor(BG)

def _norm(v):
    mn, mx = min(v), max(v)
    return [(x-mn)/(mx-mn+1e-9) for x in v]

accs   = _norm([r['final_accuracy']              for r in all_results])
f1s    = _norm([r['final_f1_macro']              for r in all_results])
spds   = _norm([1/r['training_time_s']           for r in all_results])
cpus   = _norm([1/(r['peak_cpu_memory_mb']+1)    for r in all_results])
stabs  = _norm([1/(r['loss_std']+1e-6)           for r in all_results])

for i, r in enumerate(all_results):
    c  = PALETTE.get(r['optimizer'], '#FFF')
    v  = [accs[i], f1s[i], spds[i], cpus[i], stabs[i]] + [accs[i]]
    ax.plot(angles, v, color=c, lw=2.2, label=r['optimizer'])
    ax.fill(angles, v, color=c, alpha=0.07)
    ax.scatter(angles[:-1], v[:-1], color=c, s=40, zorder=5)

ax.set_xticks(angles[:-1]); ax.set_xticklabels(dims, fontsize=11, color=TEXT_CLR)
ax.set_yticks([.25,.5,.75,1]); ax.set_yticklabels(['0.25','0.50','0.75','1.00'],
                                                    color=TEXT_CLR, fontsize=8)
ax.spines['polar'].set_color(ACCENT); ax.grid(color=GRID_CLR, lw=0.9)
ax.set_title('Optimizer Benchmark — Multi-Dimensional Radar\n(normalized per axis)',
             fontsize=13, fontweight='bold', color=ACCENT, pad=22)
ax.legend(loc='upper right', bbox_to_anchor=(1.32, 1.12))
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_radar.png'); plt.show()
print('[Plot 4 saved]')


In [ ]:
apply_theme()

# ── Plot 5: Memory & Timing bars ─────────────────────────────────────────
names   = [r['optimizer']          for r in all_results]
times   = [r['training_time_s']    for r in all_results]
cpu_mbs = [r['peak_cpu_memory_mb'] for r in all_results]
colors  = [PALETTE.get(n, '#FFF')  for n in names]
y = np.arange(len(names))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, vals, xlabel, title in [
    (ax1, times,   'Training Time (s)', 'Training Time'),
    (ax2, cpu_mbs, 'Peak CPU RAM (MB)', 'Peak CPU Memory'),
]:
    bars = ax.barh(y, vals, color=colors, edgecolor=ACCENT, linewidth=1.1)
    ax.set_yticks(y); ax.set_yticklabels(names)
    ax.set_xlabel(xlabel)
    ax.set_title(title, fontweight='bold')
    for bar, v in zip(bars, vals):
        ax.text(v*1.01, bar.get_y()+bar.get_height()/2,
                f'{v:.0f}', va='center', color=TEXT_CLR, fontsize=9)
    _spine(ax)
fig.suptitle('Optimizer Efficiency — Time & Memory',
             fontsize=14, fontweight='bold', color=ACCENT, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_memory_timing.png'); plt.show()
print('[Plot 5 saved]')


In [ ]:
apply_theme()

# ── Plot 6: Edge AI Compression Dashboard ────────────────────────────────
panels = {
    'Disk Size (MB)': (eval_report['baseline_disk_mb'],               eval_report['quantized_disk_mb']),
    'Latency (ms)':   (eval_report['baseline_latency']['mean_ms'],    eval_report['quantized_latency']['mean_ms']),
    'Accuracy':       (eval_report['baseline_accuracy'],              eval_report['quantized_accuracy']),
    'F1 Macro':       (eval_report['baseline_f1'],                    eval_report['quantized_f1']),
}
fig, axes = plt.subplots(1, 4, figsize=(18, 6))
for ax, (lbl, (bv, qv)) in zip(axes, panels.items()):
    bars = ax.bar(['Baseline\nFP32', 'Quantized\nINT8'], [bv, qv],
                  color=[BASE_CLR, QUANT_CLR], width=0.5,
                  edgecolor=ACCENT, linewidth=1.3)
    for bar, v in zip(bars, [bv, qv]):
        fmt = f'{v:.3f}' if v < 10 else f'{v:.1f}'
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.015,
                fmt, ha='center', va='bottom', color=TEXT_CLR,
                fontsize=9, fontweight='bold')
    ax.set_title(lbl, fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(bv, qv) * 1.20); _spine(ax)
fig.legend(
    handles=[Line2D([0],[0],color=BASE_CLR, lw=7,label='Baseline FP32'),
             Line2D([0],[0],color=QUANT_CLR,lw=7,label='Quantized INT8')],
    loc='lower center', ncol=2, bbox_to_anchor=(0.5,-0.04), fontsize=10)
fig.suptitle(
    f'Edge AI Report — FP32 vs INT8  |  '
    f'Speedup: {eval_report["speedup"]:.2f}x  |  '
    f'Disk compression: {eval_report["compression_ratio_disk"]:.2f}x  |  '
    f'Acc drop: {eval_report["accuracy_drop"]:+.4f}',
    fontsize=12, fontweight='bold', color=ACCENT, y=1.03)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_compression.png'); plt.show()
print('[Plot 6 saved]')
print(f'\nAll plots saved in {OUTPUT_DIR}/')


---
## Pipeline Complete

```
./results/
   benchmark_results.json      ← optimizer metrics
   edge_evaluation_report.json ← compression metrics
   plot_train_loss.png
   plot_eval_loss.png
   plot_accuracy_f1.png
   plot_radar.png
   plot_memory_timing.png
   plot_compression.png
./best_model/                   ← best FP32 checkpoint
./compressed_model/             ← INT8 weights + config
```
